# SDE-Net post-hoc analysis con label STGAN

Questo notebook **non addestra nuovamente SDE-Net o STGAN**. Legge le predizioni SDE-Net già salvate per t+1h, t+6h e t+12h, le collega alle decisioni STGAN sulla coppia esatta `(location, timestamp)` e rigenera la stessa analisi post-hoc normal/rare.

La label usata dai grafici è esclusivamente `anomaly_group` derivata da `is_anomaly` di STGAN. `event_group` non viene creato e non è accettato come fallback. Le righe iniziali eventualmente non valutate da STGAN per il suo context window sono escluse tramite inner join e conteggiate nell'audit.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in Path.cwd().resolve().parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi e configurazione

I percorsi possono essere sovrascritti sul server con `STGAN_SEED_DIR`, `SDE_PREDICTIONS_T1`, `SDE_PREDICTIONS_T6`, `SDE_PREDICTIONS_T12` e `STGAN_POSTHOC_ROOT`.

In [ ]:
FORECAST_HORIZONS = (1, 6, 12)
STGAN_SEED = 20
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'

# Stessa configurazione usata dal notebook SDE-Net principale. Gli override
# di ambiente consentono di indicare direttamente i predictions.csv del server.
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False, 'anomaly_source': 'detector',
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1, 'detector_regional_quantile': 0.975,
}
HORIZON_CONFIGS = {h: {**BASE_CONFIG, 'horizon': h} for h in FORECAST_HORIZONS}
SDE_PREDICTIONS = {
    h: Path(os.environ.get(
        f'SDE_PREDICTIONS_T{h}',
        ROOT / pipe.make_out_dir(HORIZON_CONFIGS[h]) / 'predictions.csv',
    )).resolve()
    for h in FORECAST_HORIZONS
}
EVALUATION_ROOT = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT', ROOT / 'outputs' / f'sde_stgan_pointwise_posthoc_seed{STGAN_SEED}'
)).resolve()
EVALUATION_DIRS = {h: EVALUATION_ROOT / f't_plus_{h}h' for h in FORECAST_HORIZONS}
MIN_MATCH_FRACTION = 0.90
RUN_RELABEL = True
RUN_ANALYSIS = True
ALLOW_OVERWRITE = False

print('STGAN scores:', STGAN_SCORES)
for h in FORECAST_HORIZONS:
    print(f't+{h}h source:', SDE_PREDICTIONS[h])
    print(f't+{h}h output:', EVALUATION_DIRS[h])

In [ ]:
required = [STGAN_SCORES, *SDE_PREDICTIONS.values()]
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')
for h, path in SDE_PREDICTIONS.items():
    header = set(pd.read_csv(path, nrows=0).columns)
    required_sde = {'location', 'timestamp', 'y_true'}
    if not required_sde <= header or not ({'y_pred', 'y_pred_mean'} & header):
        raise ValueError(f'Schema SDE-Net t+{h}h non compatibile: {sorted(header)}')
print('OK: schemi CSV compatibili')

## 2. Join puntuale STGAN → predizioni SDE-Net

Ogni output è evaluation-only. Le colonne MTGFlow eventualmente già presenti nelle predizioni sorgenti vengono rimosse e sostituite con la decisione STGAN. Non viene creato `event_group`.

In [ ]:
RELABEL_RESULTS = {}
if RUN_RELABEL:
    for h in FORECAST_HORIZONS:
        print(f'[pointwise join] t+{h}h')
        RELABEL_RESULTS[h] = build_pointwise_detector_evaluation(
            SDE_PREDICTIONS[h], STGAN_SCORES, EVALUATION_DIRS[h],
            detector_name='stgan', min_match_fraction=MIN_MATCH_FRACTION,
            allow_overwrite=ALLOW_OVERWRITE,
        )
else:
    print('RUN_RELABEL=False: uso gli output evaluation-only già esistenti.')

AUDIT = {}
for h in FORECAST_HORIZONS:
    metadata_path = EVALUATION_DIRS[h] / 'evaluation_source.json'
    if not metadata_path.is_file():
        raise FileNotFoundError(metadata_path)
    AUDIT[h] = json.loads(metadata_path.read_text(encoding='utf-8'))
display(pd.DataFrame(AUDIT).T[[
    'detector', 'source_prediction_rows', 'matched_rows',
    'excluded_unmatched_rows', 'match_fraction', 'normal_rows', 'rare_rows',
]])

In [ ]:
# Guardia scientifica: la post-hoc STGAN deve contenere anomaly_group e mai event_group.
for h in FORECAST_HORIZONS:
    path = EVALUATION_DIRS[h] / 'predictions.csv'
    columns = set(pd.read_csv(path, nrows=0).columns)
    assert 'anomaly_group' in columns
    assert 'event_group' not in columns
print('OK: tutti gli orizzonti usano soltanto anomaly_group STGAN')

## 3. Stessa post-hoc analysis SDE-Net per t+1h, t+6h e t+12h

In [ ]:
ANALYSIS_COMMANDS = {
    h: pipe.build_analysis_command(
        str(EVALUATION_DIRS[h]), HORIZON_CONFIGS[h],
        predictions=str(EVALUATION_DIRS[h] / 'predictions.csv'),
    )
    for h in FORECAST_HORIZONS
}
if RUN_ANALYSIS:
    for h, command in ANALYSIS_COMMANDS.items():
        print(f'[post-hoc STGAN] t+{h}h')
        subprocess.run(command, check=True, cwd=ROOT)
else:
    print('RUN_ANALYSIS=False: analisi non avviata.')

In [ ]:
RESULT_FILES = [
    'metrics_global.csv', 'metrics_by_anomaly_label.csv', 'metrics_daytime.csv',
    'daytime_bin_summary.csv', 'daytime_bin_anomaly_metrics.csv',
    'frequency_weighted_bin_summary.csv', 'uncertainty_response.csv',
    'sharpness_overview.csv',
]
for h in FORECAST_HORIZONS:
    print(f'\n## t+{h}h — label STGAN puntuale')
    for name in RESULT_FILES:
        path = EVALUATION_DIRS[h] / name
        if path.is_file():
            frame = pd.read_csv(path)
            print(name, frame.shape)
            if name in {'metrics_by_anomaly_label.csv', 'sharpness_overview.csv'}:
                display(frame)

In [ ]:
FIGURES_BY_HORIZON = {
    h: pipe.build_posthoc_figures(str(EVALUATION_DIRS[h]))
    for h in FORECAST_HORIZONS
}
COMPARISON_DIR = EVALUATION_ROOT / 'horizons_1_6_12'
HORIZON_COMPARISON_FIGURES = pipe.build_horizon_comparison_figures(
    {h: str(EVALUATION_DIRS[h]) for h in FORECAST_HORIZONS},
    COMPARISON_DIR,
)
print('figure per orizzonte:', {h: list(v) for h, v in FIGURES_BY_HORIZON.items()})
print('confronto orizzonti:', list(HORIZON_COMPARISON_FIGURES))

## Interpretazione

Le curve e le tabelle confrontano l'errore SDE-Net sui campioni che **STGAN** classifica normali o rari per la stessa località e lo stesso target timestamp. Le label sono utilizzate soltanto dopo il forecasting: non modificano il training SDE-Net e non costituiscono ground truth supervisionata. Il confronto con l'analisi MTGFlow misura quindi quanto cambiano le conclusioni al cambiare del detector non supervisionato.